In [16]:
import csv
from collections import Counter
import pandas as pd
import os

# Define the paths for the parquet files and the output csv directory
base_dir = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_parquet_review_added"
csv_dir = os.path.join(base_dir, "csv_file")

splits = ["train", "val", "test"]

for split in splits:
    parquet_path = os.path.join(base_dir, f"{split}.parquet")
    csv_path = os.path.join(csv_dir, f"{split}.csv")
    # Read the parquet file
    df = pd.read_parquet(parquet_path)
    # Save as CSV
    df.to_csv(csv_path, index=False)
    print(f"Saved {split}.csv to {csv_path}")

csv_file = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_parquet_review_added/csv_file/train.csv"

purchase_history_counts = []

with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        prompt = row.get("prompt", "")
        count = prompt.count("Purchase history")
        purchase_history_counts.append(count)

# Count the distribution
counter = Counter(purchase_history_counts)

print("Purchase history count distribution:")
for num, cnt in sorted(counter.items()):
    print(f"{num} purchase histories: {cnt} rows")

# 가장 많은 purchase history가 있는 row

# Find the maximum number of purchase histories in the dataset
max_count = max(purchase_history_counts)

# Read the CSV again and print the row(s) with the maximum purchase history count
with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        prompt = row.get("prompt", "")
        if prompt.count("Purchase history") == max_count:
            print(f"Row with max purchase history ({max_count}):")
            print(row)
            print("-" * 80)

import numpy as np

# prompt 길이(토큰 수, 여기서는 whitespace로 split) 리스트 생성
prompt_lengths = []
with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        prompt = row.get("prompt", "")
        prompt_lengths.append(len(prompt.split()))

# numpy로 사분위수 및 평균 계산
prompt_lengths_np = np.array(prompt_lengths)
mean_length = np.mean(prompt_lengths_np)
q1 = np.percentile(prompt_lengths_np, 25)
q2 = np.percentile(prompt_lengths_np, 50)
q3 = np.percentile(prompt_lengths_np, 75)
q4 = np.percentile(prompt_lengths_np, 100)

print("프롬프트 길이 통계 (단어 기준):")
print(f"평균: {mean_length:.2f}")
print(f"Q1 (25%): {q1:.2f}")
print(f"Q2 (중앙값, 50%): {q2:.2f}")
print(f"Q3 (75%): {q3:.2f}")
print(f"Q4 (최대값, 100%): {q4:.2f}")
# "Metadata: " 다음에 바로 어떤 내용이 존재하는지 확인하고, "Metadata:"와 "Previous review:" 사이에 실제 내용이 있는지 체크
metadata_contents = []
with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        prompt = row.get("prompt", "")
        # "Purchase history"가 있는 경우만 체크
        idx = 0
        while True:
            idx = prompt.find("Metadata:", idx)
            if idx == -1:
                break
            start = idx + len("Metadata:")
            # "Previous review:"가 뒤에 있는지 찾기
            end = prompt.find("Previous review:", start)
            if end == -1:
                # 마지막 purchase history일 경우, 줄 끝이나 다음 purchase history까지
                # 다음 "Purchase history"가 있으면 그 전까지, 없으면 줄 끝까지
                next_ph = prompt.find("Purchase history", start)
                if next_ph == -1:
                    end = len(prompt)
                else:
                    end = next_ph
            # Metadata: 와 Previous review: 사이의 내용 추출
            content = prompt[start:end].strip()
            metadata_contents.append(content)
            idx = end
# 통계 출력
empty_count = sum(1 for c in metadata_contents if c == "")
nonempty_count = len(metadata_contents) - empty_count
print(f"Metadata: ~ Previous review: 사이에 내용이 없는 경우: {empty_count}개")
print(f"Metadata: ~ Previous review: 사이에 내용이 있는 경우: {nonempty_count}개")
# 예시 3개 출력
print("Metadata 내용 예시 (최대 3개):")
for c in metadata_contents:
    if c:
        print(f"'{c}'")
        if metadata_contents.index(c) >= 2:
            break


Saved train.csv to /Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_parquet_review_added/csv_file/train.csv
Saved val.csv to /Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_parquet_review_added/csv_file/val.csv
Saved test.csv to /Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_parquet_review_added/csv_file/test.csv
Purchase history count distribution:
0 purchase histories: 7 rows
1 purchase histories: 2 rows
2 purchase histories: 373 rows
3 purchase histories: 194 rows
4 purchase histories: 144 rows
5 purchase histories: 60 rows
6 purchase histories: 67 rows
7 purchase histories: 40 rows
8 purchase histories: 41 rows
9 purchase histories: 31 rows
10 purchase histories: 18 rows
11 purchase histories: 15 rows
12 purchase histories: 10 rows
13 purchase histories: 8 rows
14 purchase histories: 9 rows
15 purchase histories: 1 rows
17 purchase histories: 2 rows
18 purchase histories: 1 rows
22 purchase histories: 1 rows
Row with max purchase h

In [8]:
import pandas as pd

# csv_file은 이미 위에서 정의됨
df = pd.read_csv(csv_file)
# prompt 컬럼의 예시 3개를 출력
for i, prompt in enumerate(df['prompt'].head(3)):
    print(f"Prompt example {i+1}:\n{prompt}\n{'-'*40}")



Prompt example 1:
[{'content': '<|im_start|>system\nYou are a helpful AI assistant. You first think about the reasoning process in the mind and then provide the user with the answer.<|im_end|>\n<|im_start|>user\nYou are an expert in rewriting product search queries into customer-like product reviews optimized for dense retrieval systems.\n\n# Instructions:\n# 1. Analyze the user\'s previous reviews to capture their typical tone, language, and priorities.\n# 2. Rewrite the provided product search query into an authentic, casual, and emotionally engaging review, as if you have personally used the product. \n# 3. Clearly reflect the user\'s intent and explicitly highlight specific performance details from the original query.\n\n# Below are the user\'s previous reviews:\n# ```review 1: I can set this super lightweight tent up by myself in a few minutes. The ends of the poles are more like a bolt that pop right in and stay put, which I\'m ever so grateful for. Without the rainfly, I can gaz

In [17]:
import json

# 파일 경로 정의
reviews_jsonl = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_json/filtered_review_Sports_and_Outdoors.jsonl"
meta_jsonl = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/meta_data/metadata_sports_filtered.jsonl"

# 1. 메타데이터의 item_id 집합 만들기
meta_item_ids = set()
with open(meta_jsonl, "r", encoding="utf-8") as f:
    for line in f:
        try:
            item = json.loads(line)
            item_id = item.get("item_id")
            if item_id:
                meta_item_ids.add(item_id)
        except Exception as e:
            continue

print(f"메타데이터 내 item_id 개수: {len(meta_item_ids)}")

# 2. 리뷰의 parent_asin이 모두 메타데이터에 있는지 확인
missing_asins = set()
total_reviews = 0
with open(reviews_jsonl, "r", encoding="utf-8") as f:
    for line in f:
        try:
            review = json.loads(line)
            parent_asin = review.get("parent_asin")
            total_reviews += 1
            if parent_asin and parent_asin not in meta_item_ids:
                missing_asins.add(parent_asin)
        except Exception as e:
            continue

if not missing_asins:
    print("모든 parent_asin이 메타데이터에 존재합니다.")
else:
    print(f"메타데이터에 없는 parent_asin 개수: {len(missing_asins)}")
    print("예시(최대 10개):", list(missing_asins)[:10])

print(f"총 리뷰 개수: {total_reviews}")


메타데이터 내 item_id 개수: 37628
메타데이터에 없는 parent_asin 개수: 21290
예시(최대 10개): ['B000STNST8', 'B0014419V0', 'B082LSSF65', 'B00LYY0YE8', 'B000P9GZSM', 'B00ECUIT70', 'B07FLFXYX4', 'B0199BGGVW', 'B08XJPG37H', 'B007V4VSIG']
총 리뷰 개수: 48721


In [24]:
import pandas as pd

# val.parquet 파일 경로
val_parquet_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_parquet_review_added/.parquet"

# parquet 파일 읽기
df = pd.read_parquet(val_parquet_path)

# 'reword model' 컬럼에서 'ground_truth'의 'target' 값만 추출
targets = set()
for row in df['reward_model']:
    # row가 dict가 아닐 수 있으니 예외처리
    try:
        # row가 str일 경우 dict로 변환
        if isinstance(row, str):
            row = json.loads(row)
        ground_truth = row.get('ground_truth', {})
        target = ground_truth.get('target')
        if target:
            targets.add(target)
    except Exception as e:
        continue

print(f"총 target 개수: {len(targets)}")
print("예시(최대 10개):", list(targets)[:10])

# target 값들이 meta_data/metadata_sports_filtered.jsonl 파일 안에 모두 존재하는지 확인

metadata_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/meta_data/metadata_sports_filtered.jsonl"

# metadata_sports_filtered.jsonl에서 모든 item_id 수집
metadata_item_ids = set()
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            item = json.loads(line)
            item_id = item.get("item_id")
            if item_id:
                metadata_item_ids.add(item_id)
        except Exception as e:
            continue

# targets 중 metadata에 없는 값 찾기
missing_targets = [t for t in targets if t not in metadata_item_ids]

if not missing_targets:
    print("모든 target 값이 metadata_sports_filtered.jsonl에 존재합니다.")
else:
    print(f"metadata_sports_filtered.jsonl에 없는 target 개수: {len(missing_targets)}")
    print("예시(최대 10개):", missing_targets[:10])



총 target 개수: 115
예시(최대 10개): ['B08KDZ3NVF', 'B074VF9WRW', 'B0BH8MJQKB', 'B0B5B2BF5Y', 'B07HSFNMY7', 'B08F3FQKCC', 'B0007QCOO2', 'B0C5RN7TM8', 'B07BJ2JB8M', 'B09Q38XR31']
모든 target 값이 metadata_sports_filtered.jsonl에 존재합니다.


In [25]:
# Sports_filtered3plus.json의 모든 item_id가 metadata_sports_filtered.jsonl에 있는지 확인

sports_json_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/amazon_c4/sports_json/Sports_filtered3plus.json"

# Sports_filtered3plus.json에서 모든 item_id 수집
sports_item_ids = set()
with open(sports_json_path, "r", encoding="utf-8") as f:
    try:
        sports_data = json.load(f)
        for entry in sports_data:
            item_id = entry.get("item_id")
            if item_id:
                sports_item_ids.add(item_id)
    except Exception as e:
        print("Sports_filtered3plus.json 로딩 오류:", e)

# metadata_sports_filtered.jsonl에서 모든 item_id 수집 (이미 위에서 metadata_item_ids로 수집됨)

# sports_item_ids 중 metadata에 없는 값 찾기
missing_sports_item_ids = [t for t in sports_item_ids if t not in metadata_item_ids]

if not missing_sports_item_ids:
    print("Sports_filtered3plus.json의 모든 item_id가 metadata_sports_filtered.jsonl에 존재합니다.")
else:
    print(f"metadata_sports_filtered.jsonl에 없는 Sports_filtered3plus.json의 item_id 개수: {len(missing_sports_item_ids)}")
    print("예시(최대 10개):", missing_sports_item_ids[:10])


Sports_filtered3plus.json의 모든 item_id가 metadata_sports_filtered.jsonl에 존재합니다.


/opt/anaconda3/envs/zero/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [3]:
# metadata_Sports_and_Outdoors.jsonl 파일의 상위 5개 라인(레코드) 출력

metadata_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/metadata_Sports_and_Outdoors.jsonl"

print("metadata_Sports_and_Outdoors.jsonl head (상위 5개):")
try:
    with open(metadata_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(line.strip())
except Exception as e:
    print("파일 읽기 오류:", e)

# metadata_Sports_and_Outdoors.jsonl의 컬럼(키) 목록을 확인

metadata_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/metadata_Sports_and_Outdoors.jsonl"

print("metadata_Sports_and_Outdoors.jsonl의 컬럼(키) 목록:")
try:
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            first_record = json.loads(line)
            print(list(first_record.keys()))
            break
except Exception as e:
    print("파일 읽기 오류:", e)



metadata_Sports_and_Outdoors.jsonl head (상위 5개):
{"main_category": null, "title": "Sure-Grip Zombie Wheels Low 59mm 4 Pack", "average_rating": 4.5, "rating_number": 84, "features": ["Pre-packaged in sets of 4", "Low profile 59mm x 38mm", "89a w/purple hub, 92a w/black hub, 95a w/red hub, 98a w/green hub", "Made in the U.S.A.", "Anodized Aluminum Hub"], "description": ["All Zombie wheels are made in the USA. Zombie wheels feature anodized aluminum hubs for maximum durability and precise feel while maintaining rock solid stability. This allows our unique urethane compounds to deliver all your power to the floor. Choose the Zombie combination that fits your skating style and surface. Zombie Aluminum Core – Designed in house and manufactured using state of the art machining technology. What makes this different from other aluminum cores? The Zombie core is machined from a solid billet of aluminum, using the same manufacturing processes as we use to make our famous Power Trac racing plates.

In [5]:
# /Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/meta_data/metadata_Sports_and_Outdoors.jsonl에서 parent_asin이 B0778XR2QM인 레코드 찾기

target_parent_asin = "B07K8J3YQ8"
metadata_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/meta_data/metadata_Sports_and_Outdoors.jsonl"

found = False
try:
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            if record.get("parent_asin") == target_parent_asin:
                print("parent_asin이 B0778XR2QM인 레코드:")
                print(json.dumps(record, ensure_ascii=False, indent=2))
                found = True
                break
    if not found:
        print("parent_asin이 B0778XR2QM인 레코드를 찾지 못했습니다.")
except Exception as e:
    print("파일 읽기 오류:", e)


parent_asin이 B0778XR2QM인 레코드:
{
  "main_category": "Sports & Outdoors",
  "title": "Cobra Golf 2019 King Forged CB/MB Iron Set",
  "average_rating": 4.5,
  "rating_number": 24,
  "features": [
    "5 Step Forging Process-Forged 5 times to deliver more precise iron shaping and a more refined grain structure for superior feel as demanded by the best golfers.",
    "Tungsten Tour Weighting-Reconfigured to match Rickie Fowler's iron on tour, an additional tungsten insert is strategically positioned on the sole for tour preferred accuracy and precision.",
    "CNC Milled Face & Grooves-CNC Milling delivers more precise face and grooves structures for improved spin and trajectory.",
    "Flow Set-Progressive set composition flows gradually from forgiving, cavity back long irons (3-6 iron) to muscle back short irons (7-PW) designed for precision and scoring.",
    "DBM Black Finish-Cobra's most durable black satin finish, DBM (Diamonized Black Metal) provides extreme resistance to wear for lo

In [9]:
import json

input_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/meta_data/metadata_Sports_and_Outdoors.jsonl"
output_path = "/Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/meta_data/metadata_Sports_and_Outdoors_reformatted.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        record = json.loads(line)
        # Build new record
        item_id = record.get("parent_asin", "")
        title = record.get("title", "")
        description = record.get("description", "")

        if isinstance(description, list):
            description = " ".join(str(d) for d in description)
        metadata = title + " " + description + "."
    
        new_record = {
            "item_id": item_id,
            "category": "Sports",
            "metadata": metadata
        }

        fout.write(json.dumps(new_record, ensure_ascii=False) + "\n")

print(f"변환된 파일이 {output_path}에 저장되었습니다.")


변환된 파일이 /Users/shingeunbang/RLproj/Rec-R1_3/Rec-R1/data/meta_data/metadata_Sports_and_Outdoors_reformatted.jsonl에 저장되었습니다.
